In [2]:
from ftplib import FTP
import os
import json
import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
def list_folders(ftp, path):
    items = ftp.nlst(path)
    for item in items:
        try:
            ftp.cwd(item)  # Try changing to directory
            print(f"Entering directory: {item}")
            list_folders(ftp, item)  # Recursively list subfolders
            ftp.cwd('..')  # Go back to the previous directory
        except Exception as e:
            # Handle exceptions (usually thrown if 'item' is not a directory)
            pass

In [ ]:
# Usage
ftp = FTP('ftp.pride.ebi.ac.uk')
ftp.login('anonymous', 'email@example.com')  # Use anonymous login
list_folders(ftp, '/pride/data/archive/2019/04/')
ftp.quit()

In [ ]:
# The URL you want to access
url = 'https://www.ebi.ac.uk/pride/ws/archive/v2/stats/'

# Use the requests library to send a GET request to the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Print the content of the response
    print(response.text)
    names = response.json()
else:
    # Print an error message if something went wrong
    print("Failed to retrieve data: ", response.status_code)

print(names)

In [ ]:
for name in names:
    print(name)
    url = f'https://www.ebi.ac.uk/pride/ws/archive/v2/stats/{name}'
    print(url)

    # Use the requests library to send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Print the content of the response
        print(f'Writing data for {name} to file...')

        # Parse the JSON response and write it to a file
        data = response.json()
        with open(f'{name}.json', 'w') as f:
            json.dump(data, f)
    else:
        # Print an error message if something went wrong
        print("Failed to retrieve data: ", response.status_code)

In [ ]:
filename = 'SUBMISSIONS_PER_ORGANISM.json'
dict_org = {}
with open(filename, 'r') as f:
    data = json.load(f)
    for item in data:
        dict_org[item['key']] = item['value']

# write to a new file with the data sorted by value and in descending order
sorted_data = sorted(dict_org.items(), key=lambda x: x[1], reverse=True)
filename = 'list_SUBMISSIONS_PER_ORGANISM.txt'
with open(filename, 'w') as f:
    for item in sorted_data:
        f.write(f'{item[0]}: {item[1]}\n')

In [ ]:
# Create a bar plot using the list
with open(filename, 'r') as f:
    data = f.readlines()

data = [line.strip() for line in data]

# get sum of all submissions
total = sum([int(line.split(': ')[1]) for line in data])

topN = 20
data = data[:topN]  # Get the top 10 organisms
# get sum of top 10 submissions
top_total = sum([int(line.split(': ')[1]) for line in data])

print(data)
print(f'Total submissions: {total}')
print(f'Total submissions for top {topN} organisms: {top_total}')
print(f'Percentage of top {topN} organisms: {top_total/total*100:.2f}%')

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=[x.split(': ')[0] for x in data], y=[int(x.split(': ')[1]) for x in data])
plt.xticks(rotation=90)
plt.xlabel('Organism')
plt.ylabel('Number of Submissions')
plt.title('Number of Submissions per Organism')
plt.tight_layout()
plt.show()

In [ ]:
filename = 'SUBMISSIONS_PER_MODIFICATIONS.json'
dict_org = {}
with open(filename, 'r') as f:
    data = json.load(f)
    for item in data:
        dict_org[item['key']] = item['value']

# write to a new file with the data sorted by value and in descending order
sorted_data = sorted(dict_org.items(), key=lambda x: x[1], reverse=True)
filename = 'list_SUBMISSIONS_PER_MODIFICATIONS.txt'
with open(filename, 'w') as f:
    for item in sorted_data:
        f.write(f'{item[0]}: {item[1]}\n')

In [ ]:
# Create a bar plot using the list
with open(filename, 'r') as f:
    data = f.readlines()

data = [line.strip() for line in data]

# get sum of all submissions
total = sum([int(line.split(': ')[1]) for line in data])

topN = 20
data = data[:topN]  # Get the top 10 organisms
# get sum of top 10 submissions
top_total = sum([int(line.split(': ')[1]) for line in data])

print(data)
print(f'Total submissions: {total}')
print(f'Total submissions for top {topN} modifications: {top_total}')
print(f'Percentage of top {topN} modifications: {top_total/total*100:.2f}%')

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=[x.split(': ')[0] for x in data], y=[int(x.split(': ')[1]) for x in data])
plt.xticks(rotation=90)
plt.xlabel('Organism')
plt.ylabel('Number of Submissions')
plt.title('Number of Submissions per Organism')
plt.tight_layout()
plt.show()

In [ ]:
# The URL you want to access
url = 'https://www.ebi.ac.uk/pride/ws/archive/v2/projects/PXD001379'

# Use the requests library to send a GET request to the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Print the content of the response
    print(response.text)
    # Parse the JSON response
    data = response.json()
    print(data)
else:
    # Print an error message if something went wrong
    print("Failed to retrieve data: ", response.status_code)

In [ ]:
# The URL you want to access
url = 'https://www.ebi.ac.uk/pride/ws/archive/v2/projects?pageSize=100&sortDirection=DESC&sortConditions=submissionDate'

# Use the requests library to send a GET request to the URL
response = requests.get(url)

projects = []
# Check if the request was successful
if response.status_code == 200:
    # Print the content of the response
    data = response.json()['_embedded']['projects']
    for project in data:
        projects.append(project['accession'])
    with open('projects.txt', 'w') as f:
        for project in projects:
            f.write(f'{project}\n')
else:
    # Print an error message if something went wrong
    print("Failed to retrieve data: ", response.status_code)

In [ ]:
def fetch_all_projects():
    base_url = "https://www.ebi.ac.uk/pride/ws/archive/v2/projects"
    page = 1
    page_size = 100
    all_projects = []

    while True:
        params = {
            'pageSize': page_size,
            'page': page,
            'sortDirection': 'DESC',
            'sortConditions': 'submissionDate'
        }

        response = requests.get(base_url, params=params, headers={'Accept': 'application/json'})
        
        if response.status_code != 200:
            print(f"Failed to retrieve data: {response.status_code}")
            break

        data = response.json()
        if not data:  # Assuming the API returns an empty list when there are no more projects
            break
        
        with open(f'all_projects.txt', 'a') as f:
            try:
                data_projects = data['_embedded']['projects']
                projects = []
                for project in data_projects:
                    projects.append(project['accession'])
                    f.write(f"{project['accession']}\n")
                all_projects.extend(projects)

            except Exception as e:
                print(f"Failed to retrieve data: {e}")
                break

            print(f"Fetched page {page}, total projects: {len(all_projects)}, first project: {projects[0]}")
            page += 1

    return all_projects

# Usage
projects = fetch_all_projects()
print(f"Total projects fetched: {len(projects)}")

In [ ]:
# The URL you want to access
base_url = "https://www.ebi.ac.uk/pride/ws/archive/v2/search/projects"
page = 1
page_size = 100
data_projects = pd.DataFrame()

while True:
    params = {
        'keyword': '*:*',
        'filter': 'project_submission_type==PARTIAL,project_submission_type==COMPLETE',
        'pageSize': page_size,
        'page': page,
        'sortDirection': 'DESC',
        'sortFields': 'submission_date'
    }
    response = requests.get(base_url, params=params, headers={'Accept': 'application/json'})

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
        print(f"Fetched page {page}")

        try:
            projects = data['_embedded']['compactprojects']
            print(f"Total projects: {len(data_projects)}")
            
            # Convert the data to a dictionary where accession is the key
            projects_dic = {}
            for project in projects:
                acc = project.get('accession', 'N/A')
                title = project.get('title', 'N/A')
                project_description = project.get('projectDescription', 'N/A')
                sample_processing_protocol = project.get('sampleProcessingProtocol', 'N/A')
                data_processing_protocol = project.get('dataProcessingProtocol', 'N/A')
                keywords = project.get('keywords', 'N/A')
                submission_date = project.get('submissionDate', 'N/A')
                publication_date = project.get('publicationDate', 'N/A')
                license = project.get('license', 'N/A')
                updated_date = project.get('updatedDate', 'N/A')
                submitters = project.get('submitters', 'N/A')
                labPIs = project.get('labPIs', 'N/A')
                affiliations = project.get('affiliations', 'N/A')
                instruments = project.get('instruments', 'N/A')
                organisms = project.get('organisms', 'N/A')
                organismsParts = project.get('organismsParts', 'N/A')
                diseases = project.get('diseases', 'N/A')
                references = project.get('references', 'N/A')
                sdrf = project.get('sdrf', 'N/A')
                queryScore = project.get('queryScore', 'N/A')

                projects_dic[acc] = {
                    'title': title,
                    'project_description': project_description,
                    'sample_processing_protocol': sample_processing_protocol,
                    'data_processing_protocol': data_processing_protocol,
                    'keywords': keywords,
                    'submission_date': submission_date,
                    'publication_date': publication_date,
                    'license': license,
                    'updated_date': updated_date,
                    'submitters': submitters,
                    'labPIs': labPIs,
                    'affiliations': affiliations,
                    'instruments': instruments,
                    'organisms': organisms,
                    'organismsParts': organismsParts,
                    'diseases': diseases,
                    'references': references,
                    'sdrf': sdrf,
                    'queryScore': queryScore,
                }

                with open(f'projects_search_dictionaries.txt', 'a') as f:
                    f.write(f"{acc}: {projects_dic[acc]}\n")

                row = pd.DataFrame(projects_dic).T
                data_projects = pd.append(data_projects, row)

        except Exception as e:
            print(f"Failed to retrieve data: {e}")
            break
        page += 1

    else:
        # Print an error message if something went wrong
        print("Failed to retrieve data: ", response.status_code)
        break

data_projects.to_csv('projects.tsv', index=False, sep='\t')

In [ ]:
data_projects.head()

In [ ]:
lean_data = data_projects[['keywords', 'organisms', 'sdrf']]
lean_data = lean_data.reset_index()
lean_data = lean_data.rename(columns={'index': 'accession'})
lean_data.head()

In [ ]:
all_projects = []
with open('all_projects.txt', 'r') as f:
    all_projects = f.readlines()

all_projects = [project.strip() for project in all_projects]
print(all_projects)
print(len(all_projects))

In [ ]:
dataframe_projects = pd.DataFrame()

for project in tqdm(all_projects):

    base_url = f"https://www.ebi.ac.uk/pride/ws/archive/v2/projects/{project}"
    response = requests.get(base_url)
    
    if response.status_code == 200:
        
        data = response.json()
        
        with open(f'all_projects_retrieve.txt', 'a') as f:
            json.dump(data, f)
            f.write('\n')
            
        acc = data.get('accession', 'N/A')
        for key, value in data.items():
            if key == 'accession':
                continue
            dataframe_projects.loc[acc, key] = str(value)
        
dataframe_projects.to_csv('all_projects_retrieve.tsv', sep='\t')

In [ ]:
df = pd.read_csv('all_projects_retrieve.tsv', sep='\t')
df = df.rename(columns={'Unnamed: 0': 'accession'})
df.to_csv('all_projects_retrieve.tsv', sep='\t', index=False)

In [ ]:
print(df.columns)
df.head()

In [ ]:
import json
import ast
from tqdm import tqdm

def safe_json_loads(input_str):
    try:
        # First, try to load it as regular JSON
        return json.loads(input_str)
    except json.JSONDecodeError:
        # If it fails, try fixing common JSON issues and retry
        try:
            # This will handle most cases where single quotes and unescaped double quotes are issues
            fixed_str = input_str.replace("'", "\"").replace('\"', '\\\"')
            return json.loads(fixed_str)
        except json.JSONDecodeError:
            # As a last resort, try using ast.literal_eval
            try:
                return ast.literal_eval(input_str)
            except ValueError as e:
                raise ValueError(f"Failed to parse string: {e}, Original: {input_str}")

In [ ]:
dic_types = {}
dic_types_names = {}
dic_mods = {}
dic_mods_names = {}
map_types = {}
map_mods = {}

for ind in tqdm(df.index):
    list_types = df['experimentTypes'][ind]
    list_mods = df['identifiedPTMStrings'][ind]

    try:
        list_types = safe_json_loads(list_types)
        list_mods = safe_json_loads(list_mods)
    except Exception as e:
        print(f"Failed to load data: {e}")
        print(f"Data: {list_types}")
        print(f"Data: {list_mods}")
        continue

    for typ in list_types:    
        acc = typ['accession']
        name = typ['name']
        dic_types[acc] = dic_types.get(acc, 0) + 1
        dic_types_names[name] = dic_types_names.get(name, 0) + 1
        if acc not in map_types:
            map_types[acc] = name

    for mod in list_mods:
        acc = mod['accession']
        name = mod['name']
        dic_mods[acc] = dic_mods.get(acc, 0) + 1
        dic_mods_names[name] = dic_mods_names.get(name, 0) + 1
        if acc not in map_mods:
            map_mods[acc] = name

print(dic_types)
print(dic_types_names)
print(dic_mods)
print(dic_mods_names)
print(map_types)
print(map_mods)


In [ ]:
# write sorted dictionaries to files
filename = 'experiment_types.txt'
with open(filename, 'w') as f:
    for key in dict(sorted(dic_types.items(), key=lambda x: x[1], reverse=True)):
        f.write(f'{key}: {dic_types[key]}\n')

filename = 'experiment_types_names.txt'
with open(filename, 'w') as f:
    for key in dict(sorted(dic_types_names.items(), key=lambda x: x[1], reverse=True)):
        f.write(f'{key}: {dic_types_names[key]}\n')

filename = 'modifications.txt'
with open(filename, 'w') as f:
    for key in dict(sorted(dic_mods.items(), key=lambda x: x[1], reverse=True)):
        f.write(f'{key}: {dic_mods[key]}\n')

filename = 'modifications_names.txt'
with open(filename, 'w') as f:
    for key in dict(sorted(dic_mods_names.items(), key=lambda x: x[1], reverse=True)):
        f.write(f'{key}: {dic_mods_names[key]}\n')

filename = 'map_experiment_types.txt'
with open(filename, 'w') as f:
    for key, value in map_types.items():
        f.write(f'{key}: {value}\n')

filename = 'map_modifications.txt'
with open(filename, 'w') as f:
    for key, value in map_mods.items():
        f.write(f'{key}: {value}\n')

In [ ]:
# plot histogram of experiment types
# sort the dictionary by value in descending order
print(len(dic_types_names))
dic_types_names = dict(sorted(dic_types_names.items(), key=lambda x: x[1], reverse=True))
plt.figure(figsize=(10, 6))
plt.bar(dic_types_names.keys(), dic_types_names.values())
# rotate the x-axis labels
plt.xticks(rotation=90)

In [ ]:
# do the same for the top 50 modifications
print(len(dic_mods_names))
dic_mods_names = dict(sorted(dic_mods_names.items(), key=lambda x: x[1], reverse=True))
# get the top 50 modifications
dic_mods_names = dict(list(dic_mods_names.items())[:50])
plt.figure(figsize=(10, 6))
plt.bar(dic_mods_names.keys(), dic_mods_names.values())
# rotate the x-axis labels
plt.xticks(rotation=90)


# Steps

1. Read the dataframe with the projects and associated information
2. Parse the data to get the project pride accession, type of acquisition, modifications, and organism
3. Write the data to a csv

In [ ]:
df = pd.read_csv('all_projects_retrieve.tsv', sep='\t')
out_df = pd.DataFrame()

for ind in tqdm(df.index):
    try:
        list_types = safe_json_loads(df['experimentTypes'][ind])
        list_mods = safe_json_loads(df['identifiedPTMStrings'][ind])
        list_organisms = safe_json_loads(df['organisms'][ind])
    except Exception as e:
        print(f"Failed to load data: {e} for index: {ind}")
        continue

    types = []
    type_names = []
    mods = []
    mod_names = []
    organisms = []
    organism_names = []

    for typ in list_types:
        types.append(typ['accession'])
        type_names.append(typ['name'])
    for mod in list_mods:
        mods.append(mod['accession'])
        mod_names.append(mod['name'])
    for org in list_organisms:
        organisms.append(org['accession'])
        organism_names.append(org['name'])

    out_df.loc[ind, 'accession'] = df['accession'][ind]
    out_df.loc[ind, 'title'] = df['title'][ind]
    out_df.loc[ind, 'experimentTypes'] = '|'.join(types)
    out_df.loc[ind, 'experimentTypes_names'] = '|'.join(type_names)
    out_df.loc[ind, 'identifiedPTMStrings'] = '|'.join(mods)
    out_df.loc[ind, 'identifiedPTMStrings_names'] = '|'.join(mod_names)
    out_df.loc[ind, 'organisms'] = '|'.join(organisms)
    out_df.loc[ind, 'organisms_names'] = '|'.join(organism_names)

out_df.to_csv('all_projects_retrieve_cleaned.tsv', sep='\t', index=False)

In [ ]:
org_ids = {}
org_names = {}
for ind in out_df.index:
    orgs = out_df['organisms'][ind].split('|')
    names = out_df['organisms_names'][ind].split('|')
    for org, name in zip(orgs, names):
        org_ids[org] = org_ids.get(org, 0) + 1
        org_names[name] = org_names.get(name, 0) + 1

filename = 'organism_ids.txt'
with open(filename, 'w') as f:
    for key in dict(sorted(org_ids.items(), key=lambda x: x[1], reverse=True)):
        f.write(f'{key}: {org_ids[key]}\n')

filename = 'organism_names.txt'
with open(filename, 'w') as f:
    for key in dict(sorted(org_names.items(), key=lambda x: x[1], reverse=True)):
        f.write(f'{key}: {org_names[key]}\n')

In [ ]:
for ind in out_df.index:
    types = out_df['experimentTypes'][ind].split('|')
    if 'PRIDE:0000447' in types or 'PRIDE:0000450' in types:
        out_df.loc[ind, 'experiment'] = 'DIA'
    elif 'PRIDE:0000430' in types:
        out_df.loc[ind, 'experiment'] = 'XMS'
    else:
        out_df.loc[ind, 'experiment'] = 'DDA'

In [ ]:
out_df.to_csv('all_projects_retrieve_cleaned.tsv', sep='\t', index=False)

In [3]:
# read data, each line in the txt is a json object
data = []
with open('all_projects_retrieve.txt', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [4]:
data[0]

{'accession': 'PXD049090',
 'title': 'The HisRS-like domain of GCN2 is a pseudoenzyme that can bind uncharged tRNA - HX-MS Data',
 'additionalAttributes': [{'@type': 'CvParam',
   'cvLabel': 'PRIDE',
   'accession': 'PRIDE:0000411',
   'name': 'Dataset FTP location',
   'value': 'ftp://ftp.pride.ebi.ac.uk/pride/data/archive/2024/02/PXD049090'}],
 'projectDescription': 'GCN2 is a stress response kinase that phosphorylates the translation initiation factor eIF2\uf061\uf020to inhibit general protein synthesis when activated by uncharged tRNA and stalled ribosomes. The presence of a HisRS-like domain in GCN2, normally associated with the ability to bind and aminoacylate tRNAs, led to the hypothesis that eIF2\uf061 kinase activity is regulated by the direct binding of this domain to uncharged tRNA. Here we solved the structure of the HisRS-like domain in the context of full-length GCN2 by cryoEM. Structure and function analysis shows the HisRS-like domain of GCN2 has lost tRNA charging, ATP

In [5]:
# extract the data from the json objects. We need accession, title, projectDescription, sampleProcessingProtocol, dataProcessingProtocol, keywords, submissionDate, instruments, organisms, organismsParts, diseases
# beyond that, we need to extract type of data from experimentTypes and identifiedPTMStrings
# we also need to extract the organism names and the type of experiment

# create a dictionary to store the data
data_dict = {}

for item in data:
    acc = item.get('accession', 'N/A')
    title = item.get('title', 'N/A')
    project_description = item.get('projectDescription', 'N/A')
    sample_processing_protocol = item.get('sampleProcessingProtocol', 'N/A')
    data_processing_protocol = item.get('dataProcessingProtocol', 'N/A')
    keywords = item.get('keywords', 'N/A')
    submission_date = item.get('submissionDate', 'N/A')
    instruments = item.get('instruments', 'N/A')
    organisms = item.get('organisms', 'N/A')
    organisms_parts = item.get('organismsParts', 'N/A')
    diseases = item.get('diseases', 'N/A')
    experiment_types = item.get('experimentTypes', 'N/A')
    identified_ptm_strings = item.get('identifiedPTMStrings', 'N/A')

    data_dict[acc] = {
        'title': title,
        'project_description': project_description,
        'sample_processing_protocol': sample_processing_protocol,
        'data_processing_protocol': data_processing_protocol,
        'keywords': keywords,
        'submission_date': submission_date,
        'instruments': instruments,
        'organisms': organisms,
        'organisms_parts': organisms_parts,
        'diseases': diseases,
        'experiment_types': experiment_types,
        'identified_ptm_strings': identified_ptm_strings
    }

In [14]:
# let's explore one of the entries
data_dict['PXD046325']

{'title': 'Proteomic analysis of the physiological E3 ubiquitin ligase responsible for PIN1 degradation',
 'project_description': 'Induced oncoproteins degradation provides an attractive anti-cancer modality. Activation of anaphase-promoting complex (APC/CCDH1) prevents cell cycle entry by targeting crucial mitotic proteins for degradation. Phosphorylation of its co-activator CDH1 modulates the E3 ligase activity, but little is known about its regulation after phosphorylation and how to effectively harness APC/CCDH1 activity to treat cancer. Notably, Proline-directed phosphorylation is regulated by PIN1-catalyzed cis-trans prolyl isomerization to drive tumor malignancy. However, the mechanisms controlling its protein turnover remain elusive. Through proteomic screens, we identify a reciprocal antagonism of PIN1-APC/CCDH1 mediated by domain-oriented phosphorylation-dependent dual interactions as a fundamental mechanism governing mitotic protein stability and cell cycle entry.',
 'sample